Dataset

In [1]:
import huggingface_hub
from datasets import load_dataset

dataset = load_dataset(
    "parquet",
    data_files={
        "train": "https://huggingface.co/datasets/eriktks/conll2000/resolve/f7bc5ce921058824d5a9843f6a9b6a319e0ac11b/data/train-00000-of-00001.parquet",
        "test": "https://huggingface.co/datasets/eriktks/conll2000/resolve/f7bc5ce921058824d5a9843f6a9b6a319e0ac11b/data/test-00000-of-00001.parquet",
    },
)
train_val_split = dataset["train"].train_test_split(test_size=0.1, seed=42)
train_dataset = train_val_split["train"]
val_dataset = train_val_split["test"]
test_dataset = dataset["test"]

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Train dataset columns: {train_dataset.column_names}")
print(f"Validation dataset columns: {val_dataset.column_names}")
print(f"Test dataset columns: {test_dataset.column_names}")
print(f"Train dataset first example: {train_dataset[0:5]}")



Train dataset size: 8043
Validation dataset size: 894
Test dataset size: 2013
Train dataset columns: ['id', 'tokens', 'pos_tags', 'chunk_tags']
Validation dataset columns: ['id', 'tokens', 'pos_tags', 'chunk_tags']
Test dataset columns: ['id', 'tokens', 'pos_tags', 'chunk_tags']
Train dataset first example: {'id': ['5821', '2134', '6302', '3178', '869'], 'tokens': [['Which', 'types', 'of', 'stocks', 'are', 'most', 'likely', 'to', 'qualify', '?'], ['They', 'can', 'ignore', 'the', 'S&P', '500', 'stocks', 'altogether', 'and', 'focus', 'on', 'particular', 'types', 'of', 'stocks', ',', 'such', 'as', 'smaller', 'companies', ',', 'those', 'paying', 'high', 'dividends', 'or', 'companies', 'in', 'a', 'particular', 'industry', ',', 'state', 'or', 'country', '.'], ['But', 'last', 'month', 'new', 'construction', 'in', 'all', 'types', 'of', 'homes', 'waned', ',', 'from', 'single-family', 'houses', 'to', 'large', 'apartment', 'complexes', '.'], ['The', 'mutual-fund', 'industry', 'is', 'debating', 't

Part 1

In [2]:
from collections import Counter
import re
# Convert tokens to lowercase and add numbers to <NUM> token 
def normalize_token(token):
    token = token.lower()
    # If the token contains any digit, map it to <NUM>
    if re.fullmatch(r'[\d,.\-/$%]+', token):
        return "<NUM>"
    return token
# and make <UNK> token for tokens that exist less than 3 times in the training set
token_counts = Counter()
for example in train_dataset:
    for token in example["tokens"]:
        token_counts[normalize_token(token)] += 1
min_token_count = 3
vocab = {token for token, count in token_counts.items() if count >= min_token_count}
vocab.add("<UNK>") # Ensure <UNK> is explicitly in our set
def process_sentence(tokens):
    processed = []
    for token in tokens:
        norm_tok = normalize_token(token)
        # Apply <UNK> if the normalized token isn't in our vocab
        if norm_tok in vocab:
            processed.append(norm_tok)
        else:
            processed.append("<UNK>")
    return processed

train_tokens_processed = [process_sentence(example["tokens"]) for example in train_dataset]
print(f"Processed first 5 sentences in training set: {train_tokens_processed[:5]}")
vocab_list = sorted(list(vocab))
word2idx = {word: idx for idx, word in enumerate(vocab_list)}
idx2word = {idx: word for idx, word in enumerate(vocab_list)}

# Store the vocabulary size, which is a key hyperparameter for Word2Vec and matrices
vocab_size = len(vocab_list)
print(f"Vocabulary size: {vocab_size}")



Processed first 5 sentences in training set: [['which', 'types', 'of', 'stocks', 'are', 'most', 'likely', 'to', 'qualify', '?'], ['they', 'can', '<UNK>', 'the', 's&p', '<NUM>', 'stocks', '<UNK>', 'and', 'focus', 'on', 'particular', 'types', 'of', 'stocks', '<NUM>', 'such', 'as', 'smaller', 'companies', '<NUM>', 'those', 'paying', 'high', 'dividends', 'or', 'companies', 'in', 'a', 'particular', 'industry', '<NUM>', 'state', 'or', 'country', '<NUM>'], ['but', 'last', 'month', 'new', 'construction', 'in', 'all', 'types', 'of', 'homes', '<UNK>', '<NUM>', 'from', '<UNK>', 'houses', 'to', 'large', '<UNK>', '<UNK>', '<NUM>'], ['the', 'mutual-fund', 'industry', 'is', '<UNK>', 'the', 'question', '<NUM>', 'but', 'do', "n't", 'expect', 'a', 'new', 'name', 'while', 'the', 'old', 'one', 'is', 'working', 'so', 'well', '<NUM>'], ['others', 'said', 'that', 'the', 'drop', 'continued', 'the', '<UNK>', 'correction', 'that', "'s", 'been', 'due', 'in', 'the', 'petroleum', '<UNK>', 'and', 'that', 'such', 'a

PMI

In [3]:
from collections import defaultdict
import math

def build_ppmi(sentences, window_size, min_pair_count=1):
    co_occurrence_counts = defaultdict(lambda: defaultdict(int))
    total = 0
    for tokens in sentences:
        for i, w in enumerate(tokens):
            for j in range(max(0, i - window_size), min(len(tokens), i + window_size + 1)):
                if j != i:
                    co_occurrence_counts[w][tokens[j]] += 1
                    total += 1
    word_total = {w: sum(c.values()) for w, c in co_occurrence_counts.items()}
    ppmi = defaultdict(dict)
    for w, ctx in co_occurrence_counts.items():
        for c, n in ctx.items():
            if n < min_pair_count:
                continue
            pmi = math.log2((n / total) / ((word_total[w] / total) * (word_total[c] / total)))
            if pmi > 0:
                ppmi[w][c] = pmi
    return ppmi, co_occurrence_counts

ppmi_w2, co_w2 = build_ppmi(train_tokens_processed, window_size=2)
# PMI differs from raw counts because it penalizes words just have high frequencies.

TF-IDF

In [4]:
import math
from collections import Counter, defaultdict

def build_TF_IDF(sentences, vocab):
    tf_raw = defaultdict(dict)
    df = defaultdict(int)
    N = len(sentences)

    # 1. Calculate Sublinear TF and DF in a single pass
    for doc_id, tokens in enumerate(sentences):
        token_counts = Counter(tokens)
        
        for token, count in token_counts.items():
            if token in vocab:
                # Sublinear TF: 1 + log(count)
                tf_raw[token][doc_id] = 1 + math.log(count)
                # Document Frequency: count goes up by 1 because it appeared in this doc
                df[token] += 1

    # 2. Calculate IDF and the final TF-IDF sparse matrix
    tf_idf = defaultdict(dict)
    
    for token, doc_dict in tf_raw.items():
        # Smoothed IDF to prevent division by zero
        idf_score = math.log(N / (df[token]))
        
        for doc_id, tf_score in doc_dict.items():
            tf_idf[token][doc_id] = tf_score * idf_score

    return tf_idf

tfidf_sparse = build_TF_IDF(train_tokens_processed, vocab)

cosine similarity in tf-idf

In [5]:
norms = {
    word: math.sqrt(sum(v ** 2 for v in vec.values()))
    for word, vec in tfidf_sparse.items()
}

def fast_cosine(w1, w2):
    vec1, vec2 = tfidf_sparse.get(w1), tfidf_sparse.get(w2)
    if not vec1 or not vec2 or norms[w1] == 0 or norms[w2] == 0:
        return 0.0
    
    # Iterate over the smaller dict only (skips all zeros)
    if len(vec1) > len(vec2):
        vec1, vec2 = vec2, vec1
    # looping over the smaller dictionary and doing fast lookups in the larger one
    dot = sum(v * vec2[k] for k, v in vec1.items() if k in vec2)
    return dot / (norms[w1] * norms[w2])

# since dataset is financial news we can use Financial nouns: "market", "stock", "bank", "company", "million"
probe_words = ["market", "stock", "bank", "company", "million"]

for word in probe_words:
    if word not in tfidf_sparse:
        print(f"'{word}' not in vocab, skipping.")
        continue
    scores = [
        (fast_cosine(word, other), other)
        for other in tfidf_sparse
        if other != word
    ]
    top_5 = sorted(scores, reverse=True)[:5]
    print(f"Top 5 for '{word}': {[(w, round(s, 4)) for s, w in top_5]}")
    # cosine similarity  between word vectors
    # which means words that appear in same document will have high similarity

Top 5 for 'market': [('the', 0.2229), ('<NUM>', 0.1893), ('stock', 0.1838), ('in', 0.1512), ('<UNK>', 0.1511)]
Top 5 for 'stock': [('exchange', 0.388), ('composite', 0.3666), ('trading', 0.2805), ('york', 0.2464), ('<NUM>', 0.1913)]
Top 5 for 'bank': [('basir', 0.2196), ('tan', 0.2079), ('sri', 0.2079), ('m$', 0.1902), ('bumiputra', 0.1898)]
Top 5 for 'company': [('the', 0.2609), ('<NUM>', 0.2449), ('a', 0.2108), ('said', 0.2084), ("'s", 0.1937)]
Top 5 for 'million': [('share', 0.416), ('<NUM>', 0.3907), ('net', 0.3759), ('or', 0.3416), ('cents', 0.3059)]


Word2vec

In [6]:
import numpy as np
from collections import Counter

def sigmoid(z):
    z = np.clip(z, -10, 10)
    return 1.0 / (1.0 + np.exp(-z))

def train_word2vec(sentences, word2idx, embedding_dim=50, window_size=2, num_neg=5, lr=0.025, epochs=5):
    vocab_size = len(word2idx)
    np.random.seed(42)
    
    W_target = np.random.uniform(-0.5 / embedding_dim, 0.5 / embedding_dim, (vocab_size, embedding_dim))
    W_context = np.zeros((vocab_size, embedding_dim))
    
    # Precompute unigram^0.75 negative sampling distribution
    counts = np.zeros(vocab_size)
    for tokens in sentences:
        for t in tokens:
            counts[word2idx.get(t, word2idx["<UNK>"])] += 1
    neg_probs = (counts ** 0.75) / np.sum(counts ** 0.75)
    
    for epoch in range(epochs):
        total_loss = 0.0
        total_pairs = 0
        for tokens in sentences:
            token_indices = [word2idx.get(t, word2idx["<UNK>"]) for t in tokens]
            
            for i, target_idx in enumerate(token_indices):
                start = max(0, i - window_size)
                end = min(len(token_indices), i + window_size + 1)
                
                for j in range(start, end):
                    if j == i:
                        continue
                    context_idx = token_indices[j]
                    
                    # 1. Positive sample
                    v_t = W_target[target_idx].copy()  # Save original copy
                    u_c = W_context[context_idx]
                    
                    pred = sigmoid(np.dot(v_t, u_c))
                    total_loss -= np.log(pred + 1e-10)
                    
                    error = pred - 1.0
                    grad_v_t = error * u_c             # Accumulate gradient for target
                    W_context[context_idx] -= lr * error * v_t
                    
                    # 2. Negative samples (vectorized for speed)
                    neg_indices = np.random.choice(vocab_size, num_neg, replace=True, p=neg_probs)
                    u_neg = W_context[neg_indices]     # Shape: (num_neg, embedding_dim)
                    
                    preds_neg = sigmoid(np.dot(u_neg, v_t))  # Shape: (num_neg,)
                    total_loss -= np.sum(np.log(1.0 - preds_neg + 1e-10))
                    
                    errors_neg = preds_neg             # (preds_neg - 0)
                    grad_v_t += np.dot(errors_neg, u_neg)
                    
                    # Update negative context vectors and target vector
                    W_context[neg_indices] -= lr * np.outer(errors_neg, v_t)
                    W_target[target_idx] -= lr * grad_v_t
                    total_pairs += 1
        avg_loss = total_loss / total_pairs if total_pairs > 0 else 0.0            
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {avg_loss:.4f}")
        
    return W_target, W_context

W_target, W_context = train_word2vec(train_tokens_processed, word2idx, embedding_dim=50, window_size=2, num_neg=5, lr=0.025, epochs=5)

Epoch 1/5 - Loss: 2.8482
Epoch 2/5 - Loss: 2.4397
Epoch 3/5 - Loss: 2.3582
Epoch 4/5 - Loss: 2.3086
Epoch 5/5 - Loss: 2.2705


In [7]:
# Validation loss uses the training unigram^0.75 negative-sampling distribution,
def evaluate_word2vec(sentences, word2idx, W_target, W_context, neg_probs, window_size=2, num_neg=5, seed=42):
    rng = np.random.default_rng(seed)
    vocab_size = len(word2idx)
    total_loss = 0.0
    total_pairs = 0

    for tokens in sentences:
        token_indices = [word2idx.get(t, word2idx["<UNK>"]) for t in tokens]
        for i, target_idx in enumerate(token_indices):
            start = max(0, i - window_size)
            end = min(len(token_indices), i + window_size + 1)
            for j in range(start, end):
                if j == i:
                    continue
                context_idx = token_indices[j]
                v_t = W_target[target_idx]
                positive_score = sigmoid(np.dot(v_t, W_context[context_idx]))
                total_loss -= np.log(positive_score + 1e-10)

                neg_indices = rng.choice(vocab_size, size=num_neg, replace=True, p=neg_probs)
                negative_scores = sigmoid(W_context[neg_indices] @ v_t)
                total_loss -= np.sum(np.log(1.0 - negative_scores + 1e-10))
                total_pairs += 1

    return total_loss / max(total_pairs, 1)

# Rebuild the same negative-sampling distribution from the training sentences.
train_counts = np.zeros(len(word2idx), dtype=np.float64)
for tokens in train_tokens_processed:
    for token in tokens:
        train_counts[word2idx[token]] += 1
train_neg_probs = train_counts ** 0.75
train_neg_probs /= train_neg_probs.sum()
val_tokens_processed = [process_sentence(example["tokens"]) for example in val_dataset]
val_loss = evaluate_word2vec(
    val_tokens_processed, word2idx, W_target, W_context, train_neg_probs,
    window_size=2, num_neg=5, seed=42
)
print(f"Validation negative-sampling loss per positive pair: {val_loss:.4f}")

Validation negative-sampling loss per positive pair: 2.3478


In [8]:
'''
configs = [dict(embedding_dim=50,  lr=0.025, epochs=5),
           dict(embedding_dim=50,  lr=0.05,  epochs=10),
           dict(embedding_dim=100, lr=0.05,  epochs=10)]
for cfg in configs:
    Wt, Wc = train_word2vec(train_tokens_processed, word2idx, window_size=2, num_neg=5, **cfg)
    print(cfg, "val loss:", evaluate_word2vec(val_tokens_processed, word2idx, Wt, Wc, train_neg_probs))

Epoch 1/5 - Loss: 2.8482
Epoch 2/5 - Loss: 2.4397
Epoch 3/5 - Loss: 2.3582
Epoch 4/5 - Loss: 2.3086
Epoch 5/5 - Loss: 2.2705
{'embedding_dim': 50, 'lr': 0.025, 'epochs': 5} val loss: 2.3478185511412435
Epoch 1/10 - Loss: 2.6357
Epoch 2/10 - Loss: 2.3580
Epoch 3/10 - Loss: 2.2950
Epoch 4/10 - Loss: 2.2455
Epoch 5/10 - Loss: 2.2064
Epoch 6/10 - Loss: 2.1751
Epoch 7/10 - Loss: 2.1497
Epoch 8/10 - Loss: 2.1281
Epoch 9/10 - Loss: 2.1129
Epoch 10/10 - Loss: 2.0967
{'embedding_dim': 50, 'lr': 0.05, 'epochs': 10} val loss: 2.4795887449020517
Epoch 1/10 - Loss: 2.6391
Epoch 2/10 - Loss: 2.3594
Epoch 3/10 - Loss: 2.2948
Epoch 4/10 - Loss: 2.2412
Epoch 5/10 - Loss: 2.1937
Epoch 6/10 - Loss: 2.1508
Epoch 7/10 - Loss: 2.1108
Epoch 8/10 - Loss: 2.0761
Epoch 9/10 - Loss: 2.0470
Epoch 10/10 - Loss: 2.0219
{'embedding_dim': 100, 'lr': 0.05, 'epochs': 10} val loss: 2.547705364269477 

so we will keep the best configuration: embedding_dim=50, lr=0.025, epochs=5
'''


'\nconfigs = [dict(embedding_dim=50,  lr=0.025, epochs=5),\n           dict(embedding_dim=50,  lr=0.05,  epochs=10),\n           dict(embedding_dim=100, lr=0.05,  epochs=10)]\nfor cfg in configs:\n    Wt, Wc = train_word2vec(train_tokens_processed, word2idx, window_size=2, num_neg=5, **cfg)\n    print(cfg, "val loss:", evaluate_word2vec(val_tokens_processed, word2idx, Wt, Wc, train_neg_probs))\n\nEpoch 1/5 - Loss: 2.8482\nEpoch 2/5 - Loss: 2.4397\nEpoch 3/5 - Loss: 2.3582\nEpoch 4/5 - Loss: 2.3086\nEpoch 5/5 - Loss: 2.2705\n{\'embedding_dim\': 50, \'lr\': 0.025, \'epochs\': 5} val loss: 2.3478185511412435\nEpoch 1/10 - Loss: 2.6357\nEpoch 2/10 - Loss: 2.3580\nEpoch 3/10 - Loss: 2.2950\nEpoch 4/10 - Loss: 2.2455\nEpoch 5/10 - Loss: 2.2064\nEpoch 6/10 - Loss: 2.1751\nEpoch 7/10 - Loss: 2.1497\nEpoch 8/10 - Loss: 2.1281\nEpoch 9/10 - Loss: 2.1129\nEpoch 10/10 - Loss: 2.0967\n{\'embedding_dim\': 50, \'lr\': 0.05, \'epochs\': 10} val loss: 2.4795887449020517\nEpoch 1/10 - Loss: 2.6391\nEpoc

In [9]:
# Pre-normalize all Word2Vec vectors once
norms = np.linalg.norm(W_target, axis=1, keepdims=True) # measure the length of each vector
W_norm = W_target / (norms + 1e-10) # scale each vector to have length 1,
# so that the dot product is the cosine similarity
idx2word = {idx: word for word, idx in word2idx.items()} # reverse mapping from index to word for easy lookup

def w2v_top_k(word, k=5):
    if word not in word2idx:
        return []
    idx = word2idx[word]
    sims = W_norm @ W_norm[idx]  # Cosine similarity with every word at once
    sims[idx] = -1.0             # Exclude the word itself
    if "<UNK>" in word2idx:
        sims[word2idx["<UNK>"]] = -1.0
        
    top_indices = np.argsort(sims)[::-1][:k]
    return [(idx2word[i], round(float(sims[i]), 4)) for i in top_indices]

for word in ["market", "stock", "bank", "company", "million"]:
    print(f"Word2Vec Top 5 for '{word}': {w2v_top_k(word)}")

Word2Vec Top 5 for 'market': [('exchange', 0.6941), ('markets', 0.6813), ('preferred', 0.6627), ('otc', 0.6623), ('war', 0.659)]
Word2Vec Top 5 for 'stock': [('plunge', 0.7649), ('activity', 0.758), ('exchange', 0.7389), ('tokyo', 0.7379), ('trading', 0.7321)]
Word2Vec Top 5 for 'bank': [('buy-out', 0.7916), ('bumiputra', 0.7701), ('proposed', 0.7674), ('parent', 0.7662), ('united', 0.7623)]
Word2Vec Top 5 for 'company': [('carrier', 0.8178), ('amdura', 0.7769), ('mattel', 0.7737), ('railroad', 0.7671), ('trinova', 0.7619)]
Word2Vec Top 5 for 'million': [('billion', 0.862), ('cubic', 0.7621), ('march', 0.7547), ('outstanding', 0.7522), ('feb.', 0.7359)]


Step 5: Compare the representations

In [10]:
import numpy as np

# Build row-normalized vector lookup views, without converting sparse matrices to dense.
def sparse_cosine_neighbors(vectors, query, k=5):
    q = vectors.get(query, {})
    q_norm = math.sqrt(sum(value * value for value in q.values()))
    if q_norm == 0:
        return []
    results = []
    for candidate, vec in vectors.items():
        if candidate == query:
            continue
        denom = q_norm * math.sqrt(sum(value * value for value in vec.values()))
        if denom == 0:
            continue
        # Iterate through the smaller mapping for the dot product.
        small, large = (q, vec) if len(q) <= len(vec) else (vec, q)
        score = sum(value * large.get(context, 0.0) for context, value in small.items()) / denom
        results.append((candidate, score))
    return sorted(results, key=lambda item: item[1], reverse=True)[:k]

queries = ["market", "stock", "bank", "company", "million"]
representations = {
    "PMI (window=2)": ppmi_w2,
    "TF-IDF (sentence docs)": tfidf_sparse,
}

def dense_cosine_neighbors(query, k=5):
    if query not in word2idx:
        return []
    query_idx = word2idx[query]
    scores = W_norm @ W_norm[query_idx]
    scores = scores.copy()
    scores[query_idx] = -np.inf
    scores[word2idx["<UNK>"]] = -np.inf
    best = np.argsort(scores)[::-1][:k]
    return [(idx2word[int(i)], float(scores[i])) for i in best]

for query in queries:
    print(f"\n{query}")
    for name, vectors in representations.items():
        print(f"  {name}: {[(w, round(s, 4)) for w, s in sparse_cosine_neighbors(vectors, query)]}")
    print(f"  Word2Vec: {[(w, round(s, 4)) for w, s in dense_cosine_neighbors(query)]}")

# Approximate storage: sparse coordinate entries or full dense arrays.
def sparse_stats(vectors):
    nnz = sum(len(row) for row in vectors.values())
    return nnz, nnz * (8 + 4)

pmi_nnz, pmi_bytes = sparse_stats(ppmi_w2)
tfidf_nnz, tfidf_bytes = sparse_stats(tfidf_sparse)
embedding_dim = W_target.shape[1]
w2v_params = W_target.size + W_context.size
w2v_bytes = w2v_params * W_target.dtype.itemsize
print("\nApproximate representation storage (excludes Python overhead):")
print(f"PMI: {pmi_nnz:,} nonzeros, about {pmi_bytes / 1024:.1f} KiB as value+index entries")
print(f"TF-IDF: {tfidf_nnz:,} nonzeros, about {tfidf_bytes / 1024:.1f} KiB as value+index entries")
print(f"Word2Vec: {embedding_dim}D target vectors ({W_target.nbytes / 1024:.1f} KiB for exported vectors); both trainable tables use {w2v_bytes / 1024:.1f} KiB")
print(f"Dimensions: PMI context vocabulary <= {len(vocab_list):,}; TF-IDF documents = {len(train_tokens_processed):,}; Word2Vec = {embedding_dim}")


market
  PMI (window=2): [('the', 0.1434), ('stock-market', 0.1372), ('plunge', 0.135), ('stock', 0.1343), ('participants', 0.1172)]
  TF-IDF (sentence docs): [('the', 0.2229), ('<NUM>', 0.1893), ('stock', 0.1838), ('in', 0.1512), ('<UNK>', 0.1511)]
  Word2Vec: [('exchange', 0.6941), ('markets', 0.6813), ('preferred', 0.6627), ('otc', 0.6623), ('war', 0.659)]

stock
  PMI (window=2): [('exchange', 0.2113), ('preferred', 0.1882), ('shares', 0.1454), ('convertible', 0.1411), ('market', 0.1343)]
  TF-IDF (sentence docs): [('exchange', 0.388), ('composite', 0.3666), ('trading', 0.2805), ('york', 0.2464), ('<NUM>', 0.1913)]
  Word2Vec: [('plunge', 0.7649), ('activity', 0.758), ('exchange', 0.7389), ('tokyo', 0.7379), ('trading', 0.7321)]

bank
  PMI (window=2): [('loan', 0.1574), ('subsidiary', 0.1439), ('banking', 0.1272), ('parent', 0.1236), ('of', 0.1182)]
  TF-IDF (sentence docs): [('basir', 0.2196), ('tan', 0.2079), ('sri', 0.2079), ('m$', 0.1902), ('bumiputra', 0.1898)]
  Word2Vec: [

Comparison notes

PMI captures local co-occurrence within the two-token window, so it is sparse and often returns topical associates. TF-IDF treats each sentence as a document; shared sentence usage can yield useful topical neighbors, while common words receive low IDF and may still appear in noisy contexts. Word2Vec compresses local context into a fixed 50-dimensional dense vector and can group related terms through shared contexts, with stronger generalization but less interpretability.

## Part 2 · Hidden Markov Model for POS Tagging

We reuse the CoNLL-2000 split above and keep the original case and punctuation because both provide POS cues. The HMM learns which tags start sentences, which tags follow one another, and which words each tag emits. Rare training tokens are pooled into `<UNK>` so we can estimate emission probabilities for unfamiliar words.

Additive smoothing gives every possible outcome a positive probability. The initial cutoff and smoothing strength below are starting points for validation.

In [11]:
from collections import Counter

import numpy as np

hmm_train_tokens = train_dataset["tokens"]
hmm_train_tags = train_dataset["pos_tags"]
hmm_min_word_count = 2
hmm_alpha = 0.1

In [12]:
def fit_pos_hmm(token_sentences, tag_sentences, min_word_count=2, alpha=0.1):
    # Skip empty sentences, which contribute no tag or word counts.
    word_counts = Counter(word for sentence in token_sentences for word in sentence)
    tag_to_id = {
        tag: i for i, tag in enumerate(sorted({tag for sentence in tag_sentences for tag in sentence}))
    }
    known_words = sorted(
        word for word, count in word_counts.items()
        if count >= min_word_count and word != "<UNK>"
    )
    word_to_id = {word: i for i, word in enumerate(["<UNK>", *known_words])}
    unk_id = word_to_id["<UNK>"]
    n_tags = len(tag_to_id)

    initial = np.full(n_tags, alpha, dtype=float)
    transition = np.full((n_tags, n_tags), alpha, dtype=float)
    emission = np.full((n_tags, len(word_to_id)), alpha, dtype=float)

    for words, tags in zip(token_sentences, tag_sentences, strict=True):
        if len(words) != len(tags):
            raise ValueError("Tokens and POS tags must align within each sentence.")
        if not tags:
            continue
        word_ids = [word_to_id.get(word, unk_id) for word in words]
        tag_ids = [tag_to_id[tag] for tag in tags]
        initial[tag_ids[0]] += 1
        np.add.at(transition, (tag_ids[:-1], tag_ids[1:]), 1)
        np.add.at(emission, (tag_ids, word_ids), 1)

    return {
        "tag_to_id": tag_to_id,
        "word_to_id": word_to_id,
        "training_words": set(word_counts),
        "log_initial": np.log(initial / initial.sum()),
        "log_transition": np.log(transition / transition.sum(axis=1, keepdims=True)),
        "log_emission": np.log(emission / emission.sum(axis=1, keepdims=True)),
    }

In [13]:
# Hand-check a corpus with repeated transitions and an unknown emission.
toy_hmm = fit_pos_hmm(
    [["dog", "dog", "dog"], ["runs"], []],
    [["N", "N", "N"], ["V"], []],
    alpha=1.0,
)
noun = toy_hmm["tag_to_id"]["N"]
dog = toy_hmm["word_to_id"]["dog"]
unk = toy_hmm["word_to_id"]["<UNK>"]
assert "runs" not in toy_hmm["word_to_id"]
assert "runs" in toy_hmm["training_words"]
np.testing.assert_allclose(np.exp(toy_hmm["log_initial"][noun]), 0.5)
np.testing.assert_allclose(np.exp(toy_hmm["log_transition"][noun, noun]), 3 / 4)
np.testing.assert_allclose(np.exp(toy_hmm["log_emission"][noun, dog]), 4 / 5)
np.testing.assert_allclose(np.exp(toy_hmm["log_emission"][noun, unk]), 1 / 5)

pos_hmm = fit_pos_hmm(hmm_train_tokens, hmm_train_tags, hmm_min_word_count, hmm_alpha)
for model in (toy_hmm, pos_hmm):
    for key in ("log_initial", "log_transition", "log_emission"):
        log_probs = model[key]
        assert np.isfinite(log_probs).all()
        np.testing.assert_allclose(np.exp(log_probs).sum(axis=-1), 1.0)

print("HMM probability checks passed.")
print(f"Training sentences: {len(hmm_train_tokens):,}")
print(f"Empty training sentences skipped: {sum(not words for words in hmm_train_tokens):,}")
print(f"POS tags: {len(pos_hmm['tag_to_id'])}")
print(f"Emission vocabulary (including <UNK>): {len(pos_hmm['word_to_id']):,}")

HMM probability checks passed.
Training sentences: 8,043
Empty training sentences skipped: 1
POS tags: 44
Emission vocabulary (including <UNK>): 9,111


### Viterbi decoding and evaluation

At each token, Viterbi keeps the best score for every possible ending tag. It adds the previous score, the transition log probability, and the current emission log probability. Backpointers recover the best complete tag sequence. Working in log space avoids multiplying many small probabilities.

We report token accuracy and macro precision, recall, and F1. Macro scores average the per-tag scores equally over tags appearing in the gold or predicted labels for each subset. A tag with an undefined precision or recall receives zero for that measure. The unknown-word subset contains tokens absent from the training text. We also count tokens sent to `<UNK>`, which includes rare training words.

In [14]:
def viterbi_decode(tokens, model):
    if len(tokens) == 0:
        return []

    word_to_id = model["word_to_id"]
    unk_id = word_to_id["<UNK>"]
    word_ids = [word_to_id.get(token, unk_id) for token in tokens]
    transition = model["log_transition"]
    emission = model["log_emission"]
    n_tags = len(model["tag_to_id"])
    backpointers = np.empty((len(tokens), n_tags), dtype=int)

    scores = model["log_initial"] + emission[:, word_ids[0]]
    tag_ids = np.arange(n_tags)
    for position, word_id in enumerate(word_ids[1:], start=1):
        # Each column contains the scores of all paths into the current tag.
        candidates = scores[:, None] + transition
        previous = candidates.argmax(axis=0)
        backpointers[position] = previous
        scores = candidates[previous, tag_ids] + emission[:, word_id]

    best_path = [int(scores.argmax())]
    for position in range(len(tokens) - 1, 0, -1):
        best_path.append(backpointers[position, best_path[-1]])
    id_to_tag = {i: tag for tag, i in model["tag_to_id"].items()}
    return [id_to_tag[tag_id] for tag_id in reversed(best_path)]

In [15]:
from itertools import product

def toy_path_score(tokens, tags):
    tag_ids = [toy_hmm["tag_to_id"][tag] for tag in tags]
    words = toy_hmm["word_to_id"]
    word_ids = [words.get(token, words["<UNK>"]) for token in tokens]
    score = toy_hmm["log_initial"][tag_ids[0]]
    for position, (word_id, tag_id) in enumerate(zip(word_ids, tag_ids, strict=True)):
        score += toy_hmm["log_emission"][tag_id, word_id]
        if position > 0:
            score += toy_hmm["log_transition"][tag_ids[position - 1], tag_id]
    return score

assert viterbi_decode([], toy_hmm) == []
for sample in (["dog"], ["sleeps"], ["dog", "sleeps", "dog"], ["runs", "dog"]):
    decoded = viterbi_decode(sample, toy_hmm)
    all_paths = product(toy_hmm["tag_to_id"], repeat=len(sample))
    best_score = max(toy_path_score(sample, path) for path in all_paths)
    np.testing.assert_allclose(toy_path_score(sample, decoded), best_score)

print("Viterbi matches exhaustive search on the toy sentences.")


Viterbi matches exhaustive search on the toy sentences.


In [16]:
def pos_metrics(gold, predicted):
    pairs = list(zip(gold, predicted, strict=True))
    if not pairs:
        return dict(tokens=0, accuracy=0.0, precision=0.0, recall=0.0, f1=0.0)

    gold_counts = Counter(gold)
    prediction_counts = Counter(predicted)
    correct = Counter(actual for actual, guess in pairs if actual == guess)
    tags = sorted(gold_counts.keys() | prediction_counts.keys())
    true_positive = np.array([correct[tag] for tag in tags])
    actual_total = np.array([gold_counts[tag] for tag in tags])
    predicted_total = np.array([prediction_counts[tag] for tag in tags])

    precision = true_positive / np.maximum(predicted_total, 1)
    recall = true_positive / np.maximum(actual_total, 1)
    f1 = 2 * true_positive / (actual_total + predicted_total)
    return dict(
        tokens=len(pairs),
        accuracy=float(true_positive.sum() / len(pairs)),
        precision=float(precision.mean()),
        recall=float(recall.mean()),
        f1=float(f1.mean()),
    )


def evaluate_pos_hmm(dataset, model):
    gold, predicted = [], []
    unknown_gold, unknown_predicted = [], []
    mapped_to_unk = 0
    for example in dataset:
        tokens, tags = example["tokens"], example["pos_tags"]
        guesses = viterbi_decode(tokens, model)
        gold.extend(tags)
        predicted.extend(guesses)
        for token, actual, guess in zip(tokens, tags, guesses, strict=True):
            mapped_to_unk += token not in model["word_to_id"]
            if token not in model["training_words"]:
                unknown_gold.append(actual)
                unknown_predicted.append(guess)
    return {
        "overall": pos_metrics(gold, predicted),
        "unknown": pos_metrics(unknown_gold, unknown_predicted),
        "mapped_to_unk": mapped_to_unk,
    }

In [17]:
validation = evaluate_pos_hmm(val_dataset, pos_hmm)

for group in ("overall", "unknown"):
    result = validation[group]
    print(
        f"{group.title()} ({result['tokens']:,} tokens): "
        f"accuracy={result['accuracy']:.2%}, "
        f"precision={result['precision']:.2%}, "
        f"recall={result['recall']:.2%}, F1={result['f1']:.2%}"
    )

print(f"Tokens mapped to <UNK>: {validation['mapped_to_unk']:,}")

Overall (21,369 tokens): accuracy=94.22%, precision=89.17%, recall=85.54%, F1=86.90%
Unknown (1,037 tokens): accuracy=51.88%, precision=25.66%, recall=23.51%, F1=23.46%
Tokens mapped to <UNK>: 1,606


### Select settings on validation

Compare three vocabulary cutoffs and three additive smoothing strengths. Select the model with the highest overall validation macro F1, using unknown-word F1 to break a tie. This choice precedes evaluation on the original test partition. The cutoff controls which rare training tokens are pooled into the unknown-word category.

In [18]:
candidates = []
for min_count in (1, 2, 3):
    for alpha in (0.01, 0.1, 1.0):
        model = fit_pos_hmm(hmm_train_tokens, hmm_train_tags, min_count, alpha)
        result = evaluate_pos_hmm(val_dataset, model)
        candidates.append((result["overall"]["f1"], result["unknown"]["f1"], min_count, alpha))
        print(
            f"cutoff={min_count}, alpha={alpha:g}: "
            f"accuracy={result['overall']['accuracy']:.2%}, "
            f"macro F1={result['overall']['f1']:.2%}, "
            f"unknown F1={result['unknown']['f1']:.2%}"
        )

_, _, best_min_count, best_alpha = max(candidates)
print(f"Selected cutoff={best_min_count}, alpha={best_alpha:g} by validation macro F1.")

cutoff=1, alpha=0.01: accuracy=94.54%, macro F1=82.45%, unknown F1=9.84%
cutoff=1, alpha=0.1: accuracy=94.36%, macro F1=84.32%, unknown F1=12.45%
cutoff=1, alpha=1: accuracy=90.25%, macro F1=76.49%, unknown F1=13.41%
cutoff=2, alpha=0.01: accuracy=94.64%, macro F1=88.61%, unknown F1=24.40%
cutoff=2, alpha=0.1: accuracy=94.22%, macro F1=86.90%, unknown F1=23.46%
cutoff=2, alpha=1: accuracy=90.76%, macro F1=77.50%, unknown F1=21.43%
cutoff=3, alpha=0.01: accuracy=93.67%, macro F1=87.83%, unknown F1=25.69%
cutoff=3, alpha=0.1: accuracy=93.35%, macro F1=86.11%, unknown F1=23.57%
cutoff=3, alpha=1: accuracy=90.34%, macro F1=78.03%, unknown F1=21.26%
Selected cutoff=2, alpha=0.01 by validation macro F1.


In [19]:
final_hmm = fit_pos_hmm(hmm_train_tokens, hmm_train_tags, best_min_count, best_alpha)
test_result = evaluate_pos_hmm(test_dataset, final_hmm)
print(f"Held-out test results with cutoff={best_min_count}, alpha={best_alpha:g}")
for group in ("overall", "unknown"):
    result = test_result[group]
    print(
        f"{group.title()} ({result['tokens']:,} tokens): "
        f"accuracy={result['accuracy']:.2%}, "
        f"precision={result['precision']:.2%}, "
        f"recall={result['recall']:.2%}, F1={result['f1']:.2%}"
    )
print(f"Tokens mapped to unknown: {test_result['mapped_to_unk']:,}")

Held-out test results with cutoff=2, alpha=0.01
Overall (47,377 tokens): accuracy=93.43%, precision=92.91%, recall=90.44%, F1=91.05%
Unknown (3,438 tokens): accuracy=54.77%, precision=21.51%, recall=20.24%, F1=20.16%
Tokens mapped to unknown: 4,841


In [20]:
# Count actual and predicted POS tag pairs on the held-out test set.
tag_ids = final_hmm["tag_to_id"]
confusion = np.zeros((len(tag_ids), len(tag_ids)), dtype=int)

for example in test_dataset:
    predictions = viterbi_decode(example["tokens"], final_hmm)
    for actual, predicted in zip(example["pos_tags"], predictions, strict=True):
        confusion[tag_ids[actual], tag_ids[predicted]] += 1

total = confusion.sum()
tp = confusion.diagonal()
fp = confusion.sum(axis=0) - tp
fn = confusion.sum(axis=1) - tp
tn = total - tp - fp - fn
f1_denominator = 2 * tp + fp + fn
active_tags = f1_denominator > 0
tag_f1 = 2 * tp / np.maximum(f1_denominator, 1)

print("Per-tag one-vs-rest confusion counts")
print(f"{'Tag':<6} {'TP':>8} {'FP':>8} {'FN':>8} {'TN':>8} {'F1':>8}")
tag_labels = test_dataset.features["pos_tags"].feature
for tag, i in sorted(tag_ids.items(), key=lambda item: item[1]):
    label = tag_labels.int2str(tag)
    score = f"{tag_f1[i]:.2%}" if active_tags[i] else "—"
    print(f"{label:<6} {tp[i]:>8,} {fp[i]:>8,} {fn[i]:>8,} {tn[i]:>8,} {score:>8}")

correct = tp.sum()
incorrect = total - correct
accuracy = correct / total
macro_f1 = tag_f1[active_tags].mean()
np.testing.assert_allclose(
    [total, accuracy, macro_f1],
    [test_result["overall"][key] for key in ("tokens", "accuracy", "f1")],
)

print()
print("Overall held-out test results")
print(f"{'Correct':>9} {'Incorrect':>9} {'Total':>9} {'Accuracy':>9} {'Macro F1':>9}")
print(f"{correct:>9,} {incorrect:>9,} {total:>9,} {accuracy:>9.2%} {macro_f1:>9.2%}")


Per-tag one-vs-rest confusion counts
Tag          TP       FP       FN       TN       F1
''          314        2        2   47,059   99.37%
#            11        0        0   47,366  100.00%
$           384       11        0   46,982   98.59%
(            77        0        0   47,300  100.00%
)            77        0        0   47,300  100.00%
,         2,390        0        0   44,987  100.00%
.         1,975        4        0   45,398   99.90%
:           238        2        0   47,137   99.58%
``          323        1        0   47,053   99.85%
CC        1,212        0        2   46,163   99.92%
CD        1,734      183      184   45,276   90.43%
DT        3,996       21       24   43,336   99.44%
EX           47        2        1   47,327   96.91%
FW            2        1        2   47,372   57.14%
IN        4,999       85       72   42,221   98.45%
JJ        2,683      655      281   43,758   85.15%
JJR         190       14       12   47,161   93.60%
JJS          70        1   